 ### Import libraries

In [48]:
import pandas as pd
import matplotlib as plt
import seaborn as sns
import numpy as np

In [49]:
# import dataset
df = pd.read_csv('C:/Users/malla/Downloads/Dataset_ATS_v2.csv')
df.head()

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes
1,Male,0,No,41,Yes,No,DSL,One year,25,No
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No


### Understanding the given data 

In [50]:
# finding the no of columns and rows

df.shape

(7043, 10)

There are 7043 columns and 10 rows

In [51]:
# Overall information of dataset. info() shows the column types and non-null counts
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   gender           7043 non-null   object
 1   SeniorCitizen    7043 non-null   int64 
 2   Dependents       7043 non-null   object
 3   tenure           7043 non-null   int64 
 4   PhoneService     7043 non-null   object
 5   MultipleLines    7043 non-null   object
 6   InternetService  7043 non-null   object
 7   Contract         7043 non-null   object
 8   MonthlyCharges   7043 non-null   int64 
 9   Churn            7043 non-null   object
dtypes: int64(3), object(7)
memory usage: 550.4+ KB


In [52]:
# Finding numeric variable or columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.758768
std,0.368612,24.559481,30.091650
min,0.000000,0.000000,18.000000
25%,0.000000,9.000000,36.000000
50%,0.000000,29.000000,70.000000
75%,0.000000,55.000000,90.000000
max,1.000000,72.000000,119.000000


### Check for missing or incomplete values

In [53]:
# check null vaules in the dataset
df.isna().sum()

gender             0
SeniorCitizen      0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
Contract           0
MonthlyCharges     0
Churn              0
dtype: int64

In [54]:
df.duplicated().sum()

np.int64(302)

 302 identical rows out of 7043 means that about 4.3% of dataset consists of pure redundancy. In a dataset of this size, having 302 rows that match perfectly across every single column is statistically highly unlikely to happen by chance. Also we can lose 5% of data during cleaning is acceptable.

In [55]:
# removing duplicate data from the dataset
df = df.drop_duplicates()

In [56]:
df.shape

(6741, 10)

There are 6741 rows and 10 columns after removing duplicate values. Since this dataset contains no missing values, I do not use imputation..

In [ ]:
df.

### Feature Engineering

#### Select relevant features
Our goal is to predict whether a telephone customer will churn. We use the nine available predictors because each could potentially relate to churn:


In [57]:
# Grouping categorical value in dataset
categorical_columns = df.select_dtypes(include=['object'], exclude=['int64'])
categorical_columns

,gender,Dependents,PhoneService,MultipleLines,InternetService,Contract,Churn
0,Female,No,No,No,DSL,Month-to-month,Yes
1,Male,No,Yes,No,DSL,One year,No
2,Female,Yes,Yes,No,DSL,Month-to-month,No
3,Female,No,Yes,No,DSL,One year,Yes
4,Male,No,Yes,No,Fiber optic,Month-to-month,No
...,...,...,...,...,...,...,...
7038,Male,No,Yes,Yes,DSL,One year,Yes
7039,Female,Yes,Yes,Yes,Fiber optic,One year,No
7040,Male,Yes,Yes,No,DSL,Month-to-month,No
7041,Male,No,Yes,Yes,Fiber optic,Month-to-month,Yes


In [58]:
# finding numeric value in dataset
numerical_columns =  df.select_dtypes(include=['int64'], exclude=['object'])
numerical_columns

,SeniorCitizen,tenure,MonthlyCharges
0,0,1,25
1,0,41,25
2,0,52,19
3,0,1,76
4,0,67,51
...,...,...,...
7038,0,1,95
7039,0,23,91
7040,0,12,21
7041,1,12,99


In [59]:
df_cleaned = df.copy()


### Our goal is to predict whether a telephone customer will churn. We use the nine available predictors because each could potentially relate to churn:
We exclude Churn from the input features because it is the outcome we want to predict. We can later examine feature importance to decide which predictors are genuinely useful.

In [60]:
features = [
    "gender", "SeniorCitizen", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService",
    "Contract", "MonthlyCharges"]

In [61]:
print("Number of predictor columns:", len(features))

Number of predictor columns: 9


### creating new features:

We create two extra variables without using the Churn result:

estimated_total_charges = tenure × MonthlyCharges. This approximates a customer's overall billed amount. It is not an actual historical total because monthly charges may change over time.

tenure_group puts customers into four tenure ranges (0–12, 13–24, 25–48, and 49+ months). This can help a model capture non-linear relationships, such as a different churn pattern for newer customers.
These calculations are based on individual customer values or fixed cut-offs, not statistics learnt from the test dataset.

In [62]:
df_cleaned["estimated_total_charges"] = (
    df_cleaned["tenure"] * df_cleaned["MonthlyCharges"]
)

# Group customers by how long they have been with the company.
df_cleaned["tenure_group"] = pd.cut(
    df_cleaned["tenure"],
    bins=[-1, 12, 24, 48, np.inf],
    labels=["0-12 months", "13-24 months", "25-48 months", "49+ months"]
)


In [63]:
features =features + [
    "estimated_total_charges", "tenure_group"
]

print("Number of predictors after feature engineering:", len(features))
display(df_cleaned.head())

Number of predictors after feature engineering: 11


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,No,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No,3417,49+ months


In [18]:
display(df_cleaned.head())

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,Churn,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,Yes,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,No,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,No,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,Yes,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,No,3417,49+ months


### Define X and y
y represents the dependent variable (output) that you want to predict or analyze against your features i.e X
 to
We encode only the target at this stage: No = 0 and Yes = 1 to avoid data leakage.

In [64]:
# X represent the feature varible
X = df_cleaned[features].copy()

# y represent target variable
y = df_cleaned["Churn"].map({"No": 0, "Yes": 1}).astype(int)

In [20]:
X

,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,InternetService,Contract,MonthlyCharges,estimated_total_charges,tenure_group
0,Female,0,No,1,No,No,DSL,Month-to-month,25,25,0-12 months
1,Male,0,No,41,Yes,No,DSL,One year,25,1025,25-48 months
2,Female,0,Yes,52,Yes,No,DSL,Month-to-month,19,988,49+ months
3,Female,0,No,1,Yes,No,DSL,One year,76,76,0-12 months
4,Male,0,No,67,Yes,No,Fiber optic,Month-to-month,51,3417,49+ months
...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,No,1,Yes,Yes,DSL,One year,95,95,0-12 months
7039,Female,0,Yes,23,Yes,Yes,Fiber optic,One year,91,2093,13-24 months
7040,Male,0,Yes,12,Yes,No,DSL,Month-to-month,21,252,0-12 months
7041,Male,1,No,12,Yes,Yes,Fiber optic,Month-to-month,99,1188,0-12 months


### Splitting dataset into Training set and Test sets
We splitting our dataset X and y into two groups:

X_train will contain 80% data of X dataset which will be used for training X_test will contain 20% data of X dataset which will be used for training y-train will contain 80% data of y dataset which will be used for training y-test will contain 20% data of y dataset which will be used for training

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [68]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 5392
Testing rows: 1349


### Feature encoding

Different columns need different preprocessing:

- Numerical predictors: We use StandardScaler so numerical variables have a mean of roughly 0 and standard deviation 1 in the training set.
- 
  Categorical predictors: We apply OneHotEncoder to make 0/1 indicator columns. 
- SeniorCitizen: it is already a 0/1 

A ColumnTransformer applies the correct steps to each feature group. The same fitted transformations will later be applied to test customers.

In [69]:
# The columns are grouped according to the processing they need.
numerical_features = [
    "tenure", "MonthlyCharges", "estimated_total_charges"
]

categorical_features = [
    "gender", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "Contract", "tenure_group"
]

binary_features = ["SeniorCitizen"]

### Enconding each features  types

Numerical columns need scaling because larger numbers like  tenure, monthlyCharges disproportionately dominate smaller numbers like tenure or binary flags).

Categorical columns need encoding because machine learning algorithms require numbers instead of text strings like "Fiber optic" or "Month-to-month").


#### Feature Scaling
### Fit on training data and transform the test data

We use fit_transform() once on X_train. It learns training medians, category mappings, and scaling values.

We use only transform() on X_test. Doing so would avoid inconsistent scaling and data leakage.

In [70]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

# Create scaler
scaler = StandardScaler()

# Scale numerical features
X_train_num = scaler.fit_transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

# Convert into DataFrames
X_train_num = pd.DataFrame(X_train_num, columns=numerical_features, index=X_train.index)
X_test_num = pd.DataFrame(X_test_num, columns=numerical_features, index=X_test.index)

## Categorical encoding

In [71]:
# Create encoder
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Encode categorical features
X_train_cat = encoder.fit_transform(X_train[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

# Convert into DataFrames
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_train_cat = pd.DataFrame(X_train_cat, columns=encoded_columns, index=X_train.index)
X_test_cat = pd.DataFrame(X_test_cat, columns=encoded_columns, index=X_test.index)

In [72]:
X_train_binary = X_train[binary_features].copy()
X_test_binary = X_test[binary_features].copy()

### Combining processed data

In [73]:
# Combine processed training data
X_train_processed = pd.concat([
    X_train_num,
    X_train_cat,
    X_train_binary
], axis=1)

# Combine processed testing data
X_test_processed = pd.concat([
    X_test_num,
    X_test_cat,
    X_test_binary
], axis=1)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)

# Display first five rows
X_train_processed.head()

Training data: (5392, 21)
Testing data: (1349, 21)


,tenure,MonthlyCharges,estimated_total_charges,gender_Female,gender_Male,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,...,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49+ months,SeniorCitizen
4349,0.944773,1.486201,1.689806,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
953,1.232687,-0.234838,0.609018,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
5550,0.821382,0.946267,1.168657,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
289,1.602861,1.621184,2.595844,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
3292,-0.988360,0.338842,-0.732786,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0


### Normalisation
### Normalise numerical features to 0–1

MinMaxScaler is an alternative to StandardScaler. It maps training values between 0 and 1

test values could fall outside that range if they exceed the training minimum/maximum. 
The categorical and binary columns are the same for both versions. 
We have to choose one feature set for each model, rather than applying both scalers one after the other.

In [74]:
from sklearn.preprocessing import MinMaxScaler
# Create a separate normalised version of the data
normalizer = MinMaxScaler()

# Encode numerical features
X_train_minmax = normalizer.fit_transform(X_train[numerical_features])
X_test_minmax = normalizer.transform(X_test[numerical_features])


In [75]:
# Convert into dataframes

X_train_minmax = pd.DataFrame(
    X_train_minmax, columns=numerical_features, index=X_train.index
)
X_test_minmax = pd.DataFrame(
    X_test_minmax, columns=numerical_features, index=X_test.index
)

# Combining the data

X_train_normalized = pd.concat(
    [X_train_minmax, X_train_cat, X_train_binary], axis=1
)
X_test_normalized = pd.concat(
    [X_test_minmax, X_test_cat, X_test_binary], axis=1
)



In [76]:
#Checking the shape of the combined data

print("Normalised training set:", X_train_normalized.shape)
print("Normalised testing set:", X_test_normalized.shape)
display(X_train_normalized.head())

Normalised training set: (5392, 21)
Normalised testing set: (1349, 21)


,tenure,MonthlyCharges,estimated_total_charges,gender_Female,gender_Male,Dependents_No,Dependents_Yes,PhoneService_No,PhoneService_Yes,MultipleLines_No,...,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year,tenure_group_0-12 months,tenure_group_13-24 months,tenure_group_25-48 months,tenure_group_49+ months,SeniorCitizen
4349,0.777778,0.910891,0.718954,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
953,0.875000,0.405941,0.433824,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
5550,0.736111,0.752475,0.581466,1.0,0.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0
289,1.000000,0.950495,0.957983,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0
3292,0.125000,0.574257,0.079832,0.0,1.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0


## Check the final prepared datasets

At this point, I check that both datasets have the same columns, contain only numerical values and have no missing values. These checks help identify errors before training a model.

In [77]:
# Check the columns match
print("Same columns:", X_train_processed.columns.equals(X_test_processed.columns))

# Check if missing values remain
print("Missing training values for standardise:", X_train_processed.isna().sum().sum())
print("Missing testing values for standardise:", X_test_processed.isna().sum().sum())
print("Missing training values for normalized:", X_train_normalized.isna().sum().sum())
print("Missing testing valuesfor normalized:", X_test_normalized.isna().sum().sum())


print(X_train_normalized[numerical_features].agg(["min", "max"]).round(2))


Same columns: True
Missing training values for standardise: 0
Missing testing values for standardise: 0
Missing training values for normalized: 0
Missing testing valuesfor normalized: 0
     tenure  MonthlyCharges  estimated_total_charges
min     0.0             0.0                      0.0
max     1.0             1.0                      1.0


## Exporting the traning and test datasets

In [78]:
X_train_processed.to_csv('C:/Users/malla/X_train_standardised.csv', index=False)
X_test_processed.to_csv('C:/Users/malla/ X_test_standardised.csv', index=False)

X_train_normalized.to_csv('C:/Users/malla/X_train_normalized.csv', index=False)
X_test_normalized.to_csv('C:/Users/malla/ X_test_normalized.csv', index=False)

y_train.to_csv('C:/Users/malla/y_train.csv', index=False)
y_test.to_csv('C:/Users/malla/y_test.csv', index=False)



In [83]:
# Exporting preprocessed dataset

#Attach the original churn labels to their matching processed rows

train_standardised = X_train_processed.copy()
train_standardised['Churn'] = y_train

test_standardised = X_test_processed.copy()
test_standardised['Churn'] = y_test

# Recombine the already-transformed train and test rows
# Sorting by original row index restores their dataset order.
preprocessed_standardised = pd.concat(
    [train_standardised, test_standardised]
).sort_index()

train_normalised = X_train_normalized.copy()
train_normalised['Churn'] = y_train

test_normalised = X_test_normalized.copy()
test_normalised['Churn'] = y_test

preprocessed_normalised = pd.concat(
    [train_normalised, test_normalised]
).sort_index()

In [84]:
preprocessed_normalised.to_csv('C:/Users/malla/preprocessed_normalized.csv', index=False)
preprocessed_standardised.to_csv('C:/Users/malla/ preprocessed_standardised.csv', index=False)

### Purpose of Saving the instance of scaler
a. When you .fit_transform() your training data, the scaler learns specific parameters—namely, the mean and standard deviation of each column in your training set.

b.When new, unseen customer data comes in later (or when you evaluate your model on your test set), that new data must be scaled using the exact same mean and standard deviation learned from the training data.

c. Saving the scaler and encoder ensures you can load it later using joblib.load('scaler.pkl') to transform new inputs identically.


In [79]:
import joblib

joblib.dump(scaler,'C:/Users/malla/standard_scaler.pkl')
joblib.dump(encoder,'C:/Users/malla/onehot_encoder.pkl')
joblib.dump(normalizer,'C:/Users/malla/normalizer.pkl')



['C:/Users/malla/normalizer.pkl']

Limitations to report: Without customer IDs, identical rows may represent different customers, so we have not removed the 302 exact duplicates. these require checking against the source. The estimated total charge is a proxy, not verified billing history. Feature engineering offers the model more information but must be assessed against a baseline to prove whether it improves performance.